In [29]:
!pip install mrjob pyspark --quiet

In [31]:
%%writefile mr_gender_avg.py
from mrjob.job import MRJob
import json
import glob

class MRGenderPriceAvg(MRJob):

    def mapper_init(self):
        self.files = glob.glob("/content/sample_data/*.json")

    def mapper(self, _, line):
        for filename in self.files:
            with open(filename) as f:
                for line in f:
                    try:
                        row = json.loads(line)
                        gender = row["data"]["gender"]
                        price = float(row["data"]["price"])
                        yield gender, price
                    except:
                        pass

    def reducer(self, gender, prices):
        prices = list(prices)
        yield gender, sum(prices) / len(prices)

if __name__ == "__main__":
    MRGenderPriceAvg.run()

Overwriting mr_gender_avg.py


In [33]:
with open("input_dummy.txt", "w") as f:
    f.write("start")

In [35]:
!python mr_gender_avg.py input_dummy.txt > mr_output.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/mr_gender_avg.root.20251203.135459.532498
Running step 1 of 1...
job output is in /tmp/mr_gender_avg.root.20251203.135459.532498/output
Streaming final output from /tmp/mr_gender_avg.root.20251203.135459.532498/output...
Removing temp directory /tmp/mr_gender_avg.root.20251203.135459.532498...


In [37]:
print(open("mr_output.txt").read())

"Boys"	478.3061224489796
"Girls"	404.1929824561403
"Men"	1524.64192139738
"Unisex"	989.1167222222223
"Women"	1328.3752620545074



SPARK

In [39]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz

!pip install -q findspark

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/o/openjdk-lts/openjdk-11-jre-headless_11.0.28%2b6-1ubuntu1%7e22.04.1_amd64.deb  404  Not Found [IP: 91.189.92.23 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/o/openjdk-lts/openjdk-11-jdk-headless_11.0.28%2b6-1ubuntu1%7e22.04.1_amd64.deb  404  Not Found [IP: 91.189.92.23 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?


In [40]:
import findspark
findspark.init("/content/spark-3.5.0-bin-hadoop3")

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GenderPriceSpark") \
    .master("local[*]") \
    .getOrCreate()

spark

In [41]:
df = spark.read.json("/content/sample_data/*.json")
df.show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [42]:
df2 = df.select(
    df["data.gender"].alias("gender"),
    df["data.price"].alias("price")
)

df2.show()
df2.printSchema()

+------+------+
|gender| price|
+------+------+
|   Men|1499.0|
|Unisex|1399.0|
| Women|8999.0|
|Unisex|1599.0|
|Unisex|1399.0|
|   Men|1499.0|
|Unisex|1599.0|
|Unisex|1699.0|
|   Men|2495.0|
|   Men|1899.0|
|   Men|9999.0|
|   Men|6999.0|
|   Men|1799.0|
|   Men|3495.0|
|   Men|9999.0|
|Unisex|599.01|
|   Men| 899.0|
|   Men| 599.0|
|Unisex|2495.0|
| Women| 649.0|
+------+------+
only showing top 20 rows

root
 |-- gender: string (nullable = true)
 |-- price: double (nullable = true)



In [43]:
from pyspark.sql import functions as F

mart_gender_avg = (
    df2.groupBy("gender")
       .agg(F.avg("price").alias("avg_price"))
)

mart_gender_avg.show()

+------+------------------+
|gender|         avg_price|
+------+------------------+
|   Men|  1524.64192139738|
| Women|1328.3752620545074|
|Unisex| 989.1167222222223|
| Girls| 404.1929824561403|
|  Boys| 478.3061224489796|
+------+------------------+



Сохраняем витрину для последующего использования

In [44]:
output_path = "/content/gender_price_mart"

mart_gender_avg.write.mode("overwrite").parquet(output_path)

In [45]:
check = spark.read.parquet(output_path)
check.show()

+------+------------------+
|gender|         avg_price|
+------+------------------+
|   Men|  1524.64192139738|
| Women|1328.3752620545074|
|Unisex| 989.1167222222223|
| Girls| 404.1929824561403|
|  Boys| 478.3061224489796|
+------+------------------+



Apache Airflow

In [46]:
!pip install "apache-airflow==2.9.0" \
  --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-2.9.0/constraints-3.12.txt"

In [47]:
!mkdir -p airflow/dags airflow/logs airflow/plugins
!export AIRFLOW_HOME=$(pwd)/airflow
!airflow db init

Traceback (most recent call last):
  File "/usr/local/bin/airflow", line 5, in <module>
    from airflow.__main__ import main
  File "/usr/local/lib/python3.12/dist-packages/airflow/__init__.py", line 40, in <module>
    from airflow import configuration, settings
  File "/usr/local/lib/python3.12/dist-packages/airflow/configuration.py", line 2347, in <module>
    secrets_backend_list = initialize_secrets_backends()
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/airflow/configuration.py", line 2261, in initialize_secrets_backends
    secrets_backend_cls = import_string(class_name)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/airflow/utils/module_loading.py", line 39, in import_string
    module = import_module(module_path)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(na

In [48]:
%%writefile /content/spark_gender_price_job.py
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("GenderPriceMart") \
    .master("local[*]") \
    .getOrCreate()

df = spark.read.json("/content/sample_data/*.json")

df2 = df.select(
    df["data.gender"].alias("gender"),
    df["data.price"].alias("price")
)

mart = df2.groupBy("gender").agg(F.avg("price").alias("avg_price"))

output_path = "/content/gender_price_mart"
mart.write.mode("overwrite").parquet(output_path)

print("Витрина данных обновлена:", output_path)
spark.stop()

Overwriting /content/spark_gender_price_job.py


In [49]:
%%writefile airflow/dags/gender_price_dag.py
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.bash import BashOperator

default_args = {
    'owner': 'student',
    'depends_on_past': False,
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

with DAG(
    "gender_price_daily_mart",
    default_args=default_args,
    start_date=datetime(2025, 12, 1),
    schedule_interval="@daily",
    catchup=False,
) as dag:

    run_spark = BashOperator(
        task_id="run_spark_job",
        bash_command="python /content/spark_gender_price_job.py"
    )

run_spark

Overwriting airflow/dags/gender_price_dag.py


In [23]:
!python /content/spark_gender_price_job.py

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/03 13:52:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/03 13:52:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/12/03 13:54:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: reentrant call inside <_io.BufferedReader name=3>

During handling of the above exception, another exception occurred:

Traceback (most r

In [24]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CheckMart") \
    .master("local[*]") \
    .getOrCreate()

# Читаем витрину
df_mart = spark.read.parquet("/content/gender_price_mart")

# Выводим на экран
df_mart.show()

spark.stop()

+------+------------------+
|gender|         avg_price|
+------+------------------+
|   Men|  1524.64192139738|
| Women|1328.3752620545074|
|Unisex| 989.1167222222223|
| Girls| 404.1929824561403|
|  Boys| 478.3061224489796|
+------+------------------+

